# Medical NLP Healthcare Chatbot: Query Classification & Specialist Routing

**Dataset:** [Medical Transcriptions (MTSamples)](https://www.kaggle.com/datasets/tboyle10/medicaltranscriptions) — ~4,999 real medical transcription notes labeled by medical specialty.

**Project framing:** This notebook builds the **NLU (Natural Language Understanding) engine** for a healthcare chatbot. A user describes their symptoms or medical query in free text; the trained neural network classifies which medical specialty it belongs to, and a lightweight response-generation layer turns that prediction into a natural-language chatbot reply that routes the user to the right department (or asks a clarifying question if confidence is low).

**Pipeline:**
1. Load & explore data
2. Clean text, select top-N specialties (dataset is noisy/imbalanced across ~40 classes)
3. TF-IDF vectorization
4. Feedforward Neural Network (Keras) — the chatbot's NLU engine
5. Train, evaluate, visualize
6. (Bonus) Embedding + LSTM model for comparison
7. Try it on your own custom queries
8. **Interactive chatbot demo** — talk to the bot directly

**Before you run this:** download `mtsamples.csv` from the Kaggle link above and place it in the same folder as this notebook (or update `DATA_PATH` below).


## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Embedding, LSTM, GlobalAveragePooling1D
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)
STOPWORDS = set(stopwords.words('english'))

np.random.seed(42)
tf.random.set_seed(42)

DATA_PATH = "mtsamples.csv"   # <-- update this if your file is named/located differently
TOP_N_CLASSES = 10            # keep the N most frequent specialties for a cleaner classification task


## 2. Load & Explore the Data

The raw file has columns: `Unnamed: 0` (index), `description`, `medical_specialty`, `sample_name`, `transcription`, `keywords`.

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=[c for c in df.columns if "Unnamed" in c], errors="ignore")
print("Shape:", df.shape)
df.head()


In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

# Drop rows with no transcription text or no label
df = df.dropna(subset=["transcription", "medical_specialty"]).reset_index(drop=True)
print("\nShape after dropping missing rows:", df.shape)
print("\nNumber of unique specialties:", df["medical_specialty"].nunique())


In [ ]:
plt.figure(figsize=(10, 8))
class_counts = df["medical_specialty"].value_counts()
sns.barplot(y=class_counts.index, x=class_counts.values)
plt.title("Number of Samples per Medical Specialty (full dataset)")
plt.xlabel("Count")
plt.tight_layout()
plt.show()

print(class_counts)


## 3. Reduce to Top-N Specialties

This dataset is well known to be noisy and heavily imbalanced (some specialties have 1 sample, others have 500+, and text often overlaps between related specialties). A standard approach — and a good talking point in your report — is to keep only the most frequent classes so the model has enough examples per class to learn from.

In [ ]:
top_classes = class_counts.head(TOP_N_CLASSES).index.tolist()
print("Selected specialties:", top_classes)

df_top = df[df["medical_specialty"].isin(top_classes)].reset_index(drop=True)
print("\nShape after filtering to top", TOP_N_CLASSES, "classes:", df_top.shape)

plt.figure(figsize=(9, 5))
sns.countplot(y="medical_specialty", data=df_top, order=top_classes)
plt.title(f"Class Distribution (Top {TOP_N_CLASSES} Specialties)")
plt.tight_layout()
plt.show()


## 4. Text Cleaning

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)          # remove punctuation/numbers
    text = re.sub(r"\s+", " ", text).strip()        # collapse whitespace
    tokens = [w for w in text.split() if w not in STOPWORDS and len(w) > 2]
    return " ".join(tokens)

df_top["clean_text"] = df_top["transcription"].apply(clean_text)
df_top[["medical_specialty", "clean_text"]].head()


## 5. Encode Labels & Split Data

In [ ]:
le = LabelEncoder()
y = le.fit_transform(df_top["medical_specialty"])
y_cat = to_categorical(y)
num_classes = len(le.classes_)
print("Classes:", list(le.classes_))

X_train_text, X_test_text, y_train, y_test, y_train_cat, y_test_cat = train_test_split(
    df_top["clean_text"], y, y_cat,
    test_size=0.2, random_state=42, stratify=y
)
print("Train size:", len(X_train_text), " Test size:", len(X_test_text))


## 6. Main Model: TF-IDF + Feedforward Neural Network

We convert text into TF-IDF vectors (weighted word-frequency features) and feed them into a Dense neural network. This is a strong, simple baseline for document classification.

In [ ]:
MAX_FEATURES = 5000

tfidf = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train_text).toarray()
X_test_tfidf = tfidf.transform(X_test_text).toarray()

print("TF-IDF train shape:", X_train_tfidf.shape)
print("TF-IDF test shape:", X_test_tfidf.shape)


In [ ]:
model = Sequential([
    Dense(256, activation="relu", input_shape=(MAX_FEATURES,)),
    Dropout(0.4),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(num_classes, activation="softmax")
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()


In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

history = model.fit(
    X_train_tfidf, y_train_cat,
    validation_split=0.15,
    epochs=40,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"], label="train loss")
axes[0].plot(history.history["val_loss"], label="val loss")
axes[0].set_title("Loss over epochs")
axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(history.history["accuracy"], label="train acc")
axes[1].plot(history.history["val_accuracy"], label="val acc")
axes[1].set_title("Accuracy over epochs")
axes[1].set_xlabel("Epoch"); axes[1].legend()

plt.tight_layout()
plt.show()


## 8. Evaluation on Test Set

In [ ]:
test_loss, test_acc = model.evaluate(X_test_tfidf, y_test_cat, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

y_pred = np.argmax(model.predict(X_test_tfidf), axis=1)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 9. Try It Yourself

Test the trained model on your own custom medical query text.

In [ ]:
def predict_specialty(text):
    cleaned = clean_text(text)
    vec = tfidf.transform([cleaned]).toarray()
    probs = model.predict(vec, verbose=0)[0]
    top_idx = np.argsort(probs)[::-1][:3]
    for i in top_idx:
        print(f"{le.classes_[i]:30s}  {probs[i]*100:.2f}%")

# Example queries — replace with your own
sample_queries = [
    "Patient complains of chest pain radiating to left arm, shortness of breath, and sweating.",
    "Patient presents for routine dental cleaning and check-up, no complaints of pain.",
    "Follow-up visit for management of type 2 diabetes and blood sugar monitoring.",
]

for q in sample_queries:
    print("Query:", q)
    predict_specialty(q)
    print("-" * 60)


## 10. (Bonus / Extension) Embedding + LSTM Model

An alternative deep learning approach that learns word embeddings directly from sequences (instead of TF-IDF bag-of-words), which can capture word order. Good to include in your report as a comparison — "does modeling word order improve results over bag-of-words?" 

In [ ]:
VOCAB_SIZE = 8000
MAX_LEN = 200
EMBED_DIM = 100

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train_text), maxlen=MAX_LEN, padding="post", truncating="post")
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test_text), maxlen=MAX_LEN, padding="post", truncating="post")

lstm_model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
    LSTM(64, dropout=0.3, recurrent_dropout=0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(num_classes, activation="softmax")
])

lstm_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
lstm_model.summary()


In [ ]:
lstm_history = lstm_model.fit(
    X_train_seq, y_train_cat,
    validation_split=0.15,
    epochs=15,
    batch_size=32,
    callbacks=[EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)],
    verbose=1
)

lstm_test_loss, lstm_test_acc = lstm_model.evaluate(X_test_seq, y_test_cat, verbose=0)
print(f"LSTM Test Accuracy: {lstm_test_acc:.4f}")
print(f"\nDense(TF-IDF) Test Accuracy: {test_acc:.4f}  vs  LSTM(Embedding) Test Accuracy: {lstm_test_acc:.4f}")


## Notes for Your Report

- **Dataset:** MTSamples via Kaggle (Boyle, T., 2019), ~4,999 medical transcription notes, ~40 medical specialties (noisy, imbalanced — this project narrows to the top {N} most frequent specialties).
- **Preprocessing:** lowercasing, punctuation/number removal, stopword removal, TF-IDF (unigrams+bigrams, top 5000 features).
- **Model 1 (main):** Feedforward NN — Dense(256)→Dropout→Dense(128)→Dropout→Dense(softmax), trained with early stopping.
- **Model 2 (bonus):** Embedding + LSTM, to show a sequence-aware alternative.
- **Evaluation:** accuracy, precision/recall/F1 per class, confusion matrix.
- **Limitations to mention:** class imbalance even after filtering, overlapping vocabulary between related specialties (e.g. surgery sub-specialties), small dataset size for deep learning.
- **Possible future work:** pretrained embeddings (GloVe/BioWordVec), transformer-based models (BioBERT/ClinicalBERT), data augmentation.


## 11. Interactive Healthcare Chatbot Demo

This section wraps the trained Dense TF-IDF model in a simple conversational loop, turning the classifier into the **NLU (Natural Language Understanding) engine** of a healthcare query-routing chatbot. The user types a free-text query; the chatbot classifies it and replies in natural language, either routing them to a department (high confidence) or asking a clarifying question and showing alternatives (low confidence).

In [ ]:
CONFIDENCE_THRESHOLD = 0.25  # tune this: lower = more decisive, higher = more cautious

def chatbot_reply(user_query, threshold=CONFIDENCE_THRESHOLD):
    cleaned = clean_text(user_query)
    vec = tfidf.transform([cleaned]).toarray()
    probs = model.predict(vec, verbose=0)[0]
    ranked = np.argsort(probs)[::-1]
    top_specialty = le.classes_[ranked[0]]
    top_conf = probs[ranked[0]]

    if top_conf < threshold:
        alternatives = ", ".join(le.classes_[i] for i in ranked[:3])
        return (
            f"I'm not fully certain about that one. It could relate to a few different "
            f"departments: {alternatives}. Could you share a bit more detail (e.g. main "
            f"symptom, how long it's lasted) so I can route you more accurately?"
        )
    else:
        return (
            f"Based on what you've described, this looks like it falls under the "
            f"**{top_specialty}** department (confidence: {top_conf*100:.1f}%). "
            f"I'd recommend booking a consultation there. If this is a medical emergency, "
            f"please seek immediate in-person care or call emergency services."
        )

print("Healthcare Query Routing Chatbot ready.")


### 11.1 Try a Few Sample Conversations

In [ ]:
sample_conversations = [
    "I have chest pain radiating to my left arm, shortness of breath, and sweating.",
    "I need a routine dental cleaning and check-up, no pain at all.",
    "I'm having a follow-up for type 2 diabetes and blood sugar monitoring.",
    "I twisted my ankle playing football and it's swollen.",
]

for query in sample_conversations:
    print("You: ", query)
    print("Bot: ", chatbot_reply(query))
    print("-" * 70)


### 11.2 Live Interactive Mode

Run the cell below to chat with the bot directly. Type `quit` or `exit` to stop.

In [ ]:
print("Healthcare Query Routing Chatbot  (type 'quit' to exit)")
print("=" * 55)
while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ("quit", "exit"):
        print("Bot: Take care! Goodbye.")
        break
    print("Bot:", chatbot_reply(user_input))
